# Cell 1 Install

In [2]:
!pip install transformers==4.29.2 simpletransformers --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


# Cell 2 Imports

In [3]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm

# เลิกใช้ simpletransformers แล้วมาใช้ของแท้จาก transformers แทน
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

In [4]:
!unzip opend_lst20_corpus
!unzip super-ai-engineer-ss-6-word-segmentation

Archive:  opend_lst20_corpus.zip
replace LST20_Corpus/AGREEMENT.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: Archive:  super-ai-engineer-ss-6-word-segmentation.zip
replace LST20 Annotation Guideline.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# Cell 3 โหลด LST20 และแปลง Word Character labels

In [9]:
# ═══════════════════════════════════════════════
# Cell 3 — สกัดข้อมูล LST20 เป็น Character-Level (Native Python)
# ═══════════════════════════════════════════════
import os
import glob
import pandas as pd
from tqdm.auto import tqdm

LST20_DIR = "./LST20_Corpus"
WS_LABELS = ['B_WORD', 'E_WORD', 'I_WORD']

def generate_lst20_char_level(data_dir, split):
    split_dir = os.path.join(data_dir, split)
    txt_files = sorted(glob.glob(os.path.join(split_dir, "*.txt")))

    for file_path in txt_files:
        with open(file_path, "r", encoding="utf-8") as f:
            words, labels = [], []
            for line in f:
                if line in ["\n", "\r\n"]:
                    if words:
                        yield {"words": words, "labels": labels}
                        words, labels = [], []
                    continue

                parts = line.split("\t")
                if len(parts) >= 4:
                    word = parts[0]
                    n = len(word)
                    if n == 0: continue

                    for i, ch in enumerate(word):
                        words.append(ch)
                        if n == 1 or i == 0:
                            labels.append("B_WORD")
                        elif i == n - 1:
                            labels.append("E_WORD")
                        else:
                            labels.append("I_WORD")

print("Building Hugging Face Datasets directly from files...")
# ใช้ generator ทำให้ดึง LST20 ได้เต็ม 100% โดย RAM ไม่พัง
train_dataset = Dataset.from_generator(lambda: generate_lst20_char_level(LST20_DIR, "train"))
eval_dataset = Dataset.from_generator(lambda: generate_lst20_char_level(LST20_DIR, "eval"))

print(f"Train sentences: {len(train_dataset):,}")
print(f"Eval sentences : {len(eval_dataset):,}")

Building Hugging Face Datasets directly from files...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train sentences: 63,310
Eval sentences : 5,620


# Cell 4 Tokenizer check ก่อนเทรน

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Pavarissy/phayathaibert-thainer")

# ทดสอบตัวอักษรไทยที่น่าเป็นห่วงที่สุด
test_chars = ['ก', 'า', 'ิ', 'ุ', 'ั', '้', '่', '็', '์', 'ๆ', '๐', '฿']

print("=== Tokenizer test (ถ้าออก [UNK] = ต้องเปลี่ยนโมเดล) ===")
unk_count = 0
for ch in test_chars:
    result = tokenizer.tokenize(ch)
    is_unk = '[UNK]' in result or result == []
    status = "UNK" if is_unk else "OK"
    print(f"  '{ch}' (U+{ord(ch):04X}) → {result}  {status}")
    if is_unk:
        unk_count += 1

if unk_count > 0:
    print(f"\n พบ {unk_count} UNK → แนะนำใช้ WangchanBERTa แทน:")
    print("airesearch/wangchanberta-base-att-spm-uncased")
else:
    print("\nTokenizer รับ char-level ได้ปกติ ใช้โมเดลนี้ได้เลย")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


=== Tokenizer test (ถ้าออก [UNK] = ต้องเปลี่ยนโมเดล) ===
  'ก' (U+0E01) → ['▁', 'ก']  OK
  'า' (U+0E32) → ['▁', 'า']  OK
  'ิ' (U+0E34) → ['▁', 'ิ']  OK
  'ุ' (U+0E38) → ['▁', 'ุ']  OK
  'ั' (U+0E31) → ['▁', 'ั']  OK
  '้' (U+0E49) → ['▁', '้']  OK
  '่' (U+0E48) → ['▁', '่']  OK
  '็' (U+0E47) → ['▁', '็']  OK
  '์' (U+0E4C) → ['▁', '์']  OK
  'ๆ' (U+0E46) → ['▁', 'ๆ']  OK
  '๐' (U+0E50) → ['▁', '๐']  OK
  '฿' (U+0E3F) → ['▁', '฿']  OK

Tokenizer รับ char-level ได้ปกติ ใช้โมเดลนี้ได้เลย


# Cell 5 Config Model (เหมือน SS5 แทบทุกอย่าง เปลี่ยน labels)

In [7]:
from transformers import TrainingArguments

# Map labels to IDs for the model
label_list = WS_LABELS
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# Cell 6 Train

In [11]:
from transformers import AutoModelForTokenClassification, Trainer, DataCollatorForTokenClassification
import torch

# 1. เช็คว่า Dataset พร้อมใช้งาน (สมมติว่ามาจาก Dataset.from_generator ใน Cell 3 แล้ว)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

# 2. Tokenize function
label2id = {l: i for i, l in enumerate(WS_LABELS)}

def tokenize_and_align_labels(examples):
    # examples["words"] คือ list ของตัวอักษร
    tokenized_inputs = tokenizer(examples["words"], truncation=True, is_split_into_words=True, max_length=510)
    labels = []

    for i, label_list in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens ให้เป็น -100 (ไม่นำมาคิด Loss)
            if word_idx is None:
                label_ids.append(-100)
            # ตัวอักษรแรกของ Subword
            elif word_idx != previous_word_idx:
                # แปลง String Label ("B_WORD", "I_WORD") เป็น ID (0, 1, 2)
                label_ids.append(label2id[label_list[word_idx]])
            # Subword ตัวตามหลัง ให้เป็น -100 ไปด้วย (เพราะเราทำ Char-level)
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Mapping datasets (Multi-core processing)...")
# ใช้ num_proc เพื่อเร่งความเร็ว และลบคอลัมน์เก่าทิ้ง
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True, num_proc=4, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(tokenize_and_align_labels, batched=True, num_proc=4, remove_columns=eval_dataset.column_names)

# 3. Training Args Adjustment
training_args.per_device_train_batch_size = 4
training_args.per_device_eval_batch_size = 4
training_args.gradient_accumulation_steps = 16
training_args.fp16 = torch.cuda.is_available() # ใช้ Mixed Precision ถ้ามีการ์ดจอ

# 4. Initialize Model
model = AutoModelForTokenClassification.from_pretrained(
    "Pavarissy/phayathaibert-thainer",
    num_labels=len(WS_LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# 5. Trainer Setup
data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

Train dataset size: 63310
Eval dataset size: 5620
Mapping datasets (Multi-core processing)...


Map (num_proc=4):   0%|          | 0/63310 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/5620 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: Pavarissy/phayathaibert-thainer
Key               | Status   |                                                                                      
------------------+----------+--------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([36]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([36, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6, 'bos_token_id': 5}.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Cell 7 Load best model & eval

In [ ]:
# Cell 7 — Evaluation using Hugging Face Trainer
metrics = trainer.evaluate()

print("--- Evaluation Results ---")
for key, value in metrics.items():
    print(f"{key}: {value}")

# Cell 8 โหลด ws_test.txt และเตรียม input

In [ ]:
with open("/kaggle/input/.../ws_test.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# CHARS_TO_EXCLUDE จะรู้จากผลการแกะรอยข้างบน
# เบื้องต้นใส่ทุก whitespace ก่อน แล้วค่อย tune
import unicodedata

def should_exclude(ch):
    """Return True ถ้า char นี้ไม่ควรอยู่ใน submission"""
    if ch == ' ':           return True
    if ch == '\n':          return True
    if ch == '\t':          return True
    if ch == '\u200b':      return True  # zero-width space
    if ch == '\xa0':        return True  # non-breaking space
    # ดัก whitespace อื่นๆ ด้วย unicode category
    if unicodedata.category(ch) == 'Zs': return True  # separator, space
    return False

char_list   = []
char_indices = []  # เก็บ index เดิมไว้ debug

for i, ch in enumerate(raw_text):
    if not should_exclude(ch):
        char_list.append(ch)
        char_indices.append(i)

target_rows = len(pd.read_csv("/kaggle/input/.../ws_sample_submission.csv"))

# ถ้าไม่เท่า → หยุดทันที ไม่ใช่แค่ warning
assert len(char_list) == target_rows, (
    f"STOP! char_list = {len(char_list):,} แต่ submission = {target_rows:,} "
    f"(ต่างกัน {abs(len(char_list) - target_rows)} ตัว) "
    f"กลับไปแก้ should_exclude() ก่อน"
)

print(f"char_list = {len(char_list):,} ตรงกับ submission เป๊ะ")

# Cell 9 Split + Predict (เหมือน pattern SS5)

In [ ]:
def split_into_chunks(chars, chunk_size=200):
    return [chars[i:i+chunk_size] for i in range(0, len(chars), chunk_size)]

chunks = split_into_chunks(char_list, 200)
predictions, _ = model.predict(chunks, split_on_space=False)

# Flatten predictions
final_labels = []
for sentence_preds in predictions:
    for token_dict in sentence_preds:
        for _, tag in token_dict.items():
            final_labels.append(tag)

print(f"Predicted labels: {len(final_labels):,}")
print(set(final_labels))

# Cell 10 Post-processing (Rule-based correction ตาม Roadmap)

In [ ]:
def fix_label_sequence(labels):
    """
    ดักจับ label ที่ผิด logic เช่น:
    - I_W หรือ E_W ที่โผล่มาโดยไม่มี B_W นำหน้า → แปลงเป็น B_W
    - B_W ตามด้วย B_W ทันที (คำ 1 ตัวที่ model ลืม assign S_W)
    """
    fixed = labels[:]
    for i in range(len(fixed)):
        if fixed[i] in ('I_W', 'E_W'):
            if i == 0 or fixed[i-1] not in ('B_W', 'I_W'):
                fixed[i] = 'B_W'
        if fixed[i] == 'B_W' and i + 1 < len(fixed):
            if fixed[i+1] == 'B_W':
                fixed[i] = 'S_W'
    return fixed

final_labels = fix_label_sequence(final_labels)

# Cell 11 สร้าง Submission

In [ ]:
assert len(final_labels) == len(submission), (
    f"final_labels ({len(final_labels):,}) ≠ submission ({len(submission):,})\n"
    f"ห้ามตัด/เติม label มั่วๆ → กลับไปแก้ Cell 8"
)

submission['label'] = final_labels
submission.to_csv("submission.csv", index=False)

# Sanity check label values
unexpected = set(final_labels) - set(WS_LABELS)
assert not unexpected, f"❌ พบ label ที่ไม่ควรมี: {unexpected}"

print("Submission พร้อมส่ง")
print(submission['label'].value_counts())